# Notebook 5: Regime Analysis

The rolling-window plot in Notebook 4 suggests the VIX-to-CEE coefficient isn't constant; it appears to deepen during high-volatility episodes like the COVID crash and the Russia-Ukraine shock. But "appears to deepen" is just an observation, not a statistical proof. This notebook observes the statistical proof by splitting the sample into fear regimes and testing whether the VIX effect is statistically different between them.

I use two regime definitions to make sure the result doesn't depend on a particular threshold choice.

**Crisis regime: VIX > 25.** This is the conventional fear threshold in the practitioner literature. The CBOE itself describes VIX above 25 as "elevated volatility." Roughly the top 19% of trading days in my sample fall in this regime, concentrated heavily in 2020 and 2022.

**Median split: VIX above versus below 18.12.** This is a balanced 50/50 split using the sample median, and serves as a robustness check.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path

Path("../figures").mkdir(exist_ok=True)

df = pd.read_parquet("../data/regression_data.parquet")
print(f"Observations: {len(df)}")
print(f"Date range:   {df.index[0].date()} to {df.index[-1].date()}")
df.head()

Observations: 1635
Date range:   2018-01-04 to 2024-12-23


,VIX_level,VIX_change,WIG20_next,BUX_next,PX_next,ATX_next
Date,,,,,,
2018-01-04,9.22,0.070001,-0.001430,0.003636,0.000172,0.000124
2018-01-05,9.22,0.000000,0.007507,0.002718,0.003252,-0.000183
2018-01-08,9.52,0.300000,-0.007969,-0.005864,-0.004012,0.002336
2018-01-09,10.08,0.559999,-0.006790,-0.005491,-0.002493,0.009308
2018-01-10,9.82,-0.260000,0.009231,0.003060,0.005386,0.002765


## Load the Data

Same regression-ready dataset from Notebook 1.

In [4]:
# define crisis regime: VIX > 25 (standard "fear threshold" in the literature)
# also create a sample-median regime for robustness
threshold_crisis = 25
threshold_median = df["VIX_level"].median()

df["regime_crisis"] = (df["VIX_level"] > threshold_crisis).astype(int)
df["regime_median"] = (df["VIX_level"] > threshold_median).astype(int)

# count observations in each regime
print("Regime definitions:\n")
print(f"  Crisis threshold (VIX > 25):")
print(f"    High-fear days: {df['regime_crisis'].sum()} ({df['regime_crisis'].mean()*100:.1f}%)")
print(f"    Low-fear days:  {(1-df['regime_crisis']).sum()} ({(1-df['regime_crisis'].mean())*100:.1f}%)")
print()
print(f"  Median threshold (VIX > {threshold_median:.2f}):")
print(f"    High-fear days: {df['regime_median'].sum()} ({df['regime_median'].mean()*100:.1f}%)")
print(f"    Low-fear days:  {(1-df['regime_median']).sum()} ({(1-df['regime_median'].mean())*100:.1f}%)")

# brief check: what dates fall in the crisis regime?
crisis_dates = df[df["regime_crisis"] == 1].index
print(f"\nFirst crisis day:  {crisis_dates[0].date()}")
print(f"Last crisis day:   {crisis_dates[-1].date()}")
print(f"\nMajor crisis clusters (by year):")
print(crisis_dates.to_series().dt.year.value_counts().sort_index())

Regime definitions:

  Crisis threshold (VIX > 25):
    High-fear days: 304 (18.6%)
    Low-fear days:  1331 (81.4%)

  Median threshold (VIX > 18.12):
    High-fear days: 817 (50.0%)
    Low-fear days:  818 (50.0%)

First crisis day:  2018-02-05
Last crisis day:   2024-12-18

Major crisis clusters (by year):
Date
2018     13
2019      1
2020    145
2021     20
2022    119
2023      2
2024      4
Name: count, dtype: int64


## Define the Fear Regimes

Two binary dummy variables, one for each threshold definition.

The crisis dummy uses VIX > 25. I print out the distribution of crisis days by year to verify that the dummy captures the actual stress episodes (COVID in 2020, Russia-Ukraine in 2022) and not something else.

The median dummy splits the sample symmetrically. This is purely a robustness check: if the regime effect only shows up under the crisis threshold, it might be driven by a few extreme observations rather than a genuine regime difference.

In [ ]:
def run_interaction_ols(df, market, regime_col, lags=5):
    """
    OLS regression with VIX_change × regime interaction.
    Returns the fitted model.
    """
    y = df[f"{market}_next"]
    
    X = pd.DataFrame({
        "VIX_level"   : df["VIX_level"],
        "VIX_change"  : df["VIX_change"],
        "Regime"      : df[regime_col],
        "VIXch_x_Reg" : df["VIX_change"] * df[regime_col],
    })
    
    X_with_const = sm.add_constant(X)
    model = sm.OLS(y, X_with_const)
    return model.fit(cov_type="HAC", cov_kwds={"maxlags": lags})

# quick test on PX (highest baseline R²)
test_res = run_interaction_ols(df, "PX", "regime_crisis")
print("Test interaction regression: PX, crisis regime (VIX > 25)\n")
print(test_res.summary().tables[1])
print(f"\nR² = {test_res.rsquared:.4f}")